# __Find nodes that serve as candidates for connectors__
- keep only main and valid links
- determine their priority based on their highway type
- from/to nodes on those links are candidates for connector creation
- connectors search for near links 
- they will only consider nodes from main links as candidates to create connectors
- these notebook identifies these nodes and tags them
- nodes with priority != 0 are nodes from main avenues that will be considered for connectors

In [1]:
import win32com.client as com
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import numpy as np
from functools import lru_cache
from collections import defaultdict
from shapely.geometry import LineString
from shapely import wkt
import seaborn as sns

### __Read Visum .ver__

In [2]:
#Red base GDL (sin links agregados por Johan y Daniel)
jeannette_ver = r'C:\Users\AP03542515\OneDrive - Instituto Tecnologico y de Estudios Superiores de Monterrey\Modelación Urbana - Red Vial Guadalajara\Visum Projects\Jeannette\RedOSMNX_AMG_24 - Jul.ver'

import win32com.client
Visum = com.Dispatch("Visum.Visum") #Visum 24 version
Visum.LoadVersion(jeannette_ver)
C = win32com.client.constants

### __Read Links__

In [3]:
# === 1. Retrieve links ===
links = pd.DataFrame({
    "No": [i[1] for i in Visum.Net.Links.GetMultiAttValues("No")],
    "FromNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("FromNodeNo")],
    "ToNodeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("ToNodeNo")],
    "TypeNo": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TypeNo")],
    "TSysSet": [i[1] for i in Visum.Net.Links.GetMultiAttValues("TSysSet")],
    "Length": [i[1] for i in Visum.Net.Links.GetMultiAttValues("Length")],
    "U": [i[1] for i in Visum.Net.Links.GetMultiAttValues("U")],
    "V": [i[1] for i in Visum.Net.Links.GetMultiAttValues("V")],
    "KEY": [i[1] for i in Visum.Net.Links.GetMultiAttValues("KEY")],
    "HIGHWAY": [i[1] for i in Visum.Net.Links.GetMultiAttValues("HIGHWAY")],
    "CAPACIDAD_FINAL": [i[1] for i in Visum.Net.Links.GetMultiAttValues("CAPACIDAD_FINAL")],
    "CARRILES_FINAL": [i[1] for i in Visum.Net.Links.GetMultiAttValues("CARRILES_FINAL")],
    "VELPROM_FINAL": [i[1] for i in Visum.Net.Links.GetMultiAttValues("VELPROM_FINAL")],
    "LIMVEL_FINAL": [i[1] for i in Visum.Net.Links.GetMultiAttValues("LIMVEL_FINAL")],
    "geometry": [i[1] for i in Visum.Net.Links.GetMultiAttValues("WKTPolyWGS84")],
})

links['geometry'] = links['geometry'].apply(wkt.loads)
links = gpd.GeoDataFrame(links, geometry='geometry', crs="EPSG:4326")
print(f"Read {len(links):,} from Visum")


Read 593,968 from Visum


# __Simplify network__

### 1. Remove links without attributes 
those are the ones with invalid highways and include:
- main invalid highways (pedestrian, cycleway, TL, busway etc)
- phantom pair links with no TsysSet

In [4]:
mask_has_atts = (
    (links['CAPACIDAD_FINAL'] > 0) &
    (links['CARRILES_FINAL'] > 0) &
    (links['LIMVEL_FINAL'] > 0)
)

# links with no atts
links_no_atts = links[~mask_has_atts].copy()

#links with atts
links_with_atts = links[mask_has_atts].copy()

print(f"Highways of links with no attributes: {links_no_atts['HIGHWAY'].unique().tolist()}")
print()
print(f"Highways of links with attributes: {links_with_atts['HIGHWAY'].unique().tolist()}")


Highways of links with no attributes: ['', 'path', "['footway', 'residential']", 'footway', "['service', 'pedestrian']", "['pedestrian', 'service']", "['living_street', 'footway']", "['living_street', 'residential']", 'pedestrian', "['path', 'residential']", "['living_street', 'path']", "['track', 'residential']", "['pedestrian', 'residential']", "['footway', 'steps']", "['service', 'steps']", "['service', 'residential']", 'steps', 'track', "['unclassified', 'residential']", "['service', 'path']", "['steps', 'residential']", "['track', 'path']", "['living_street', 'track']", "['footway', 'path', 'steps']", "['track', 'service']", "['path', 'unclassified']", "['track', 'unclassified']", 'cycleway', "['footway', 'service']", "['footway', 'path']", "['living_street', 'steps']", "['pedestrian', 'steps']", 'busway', "['footway', 'service', 'residential']", "['footway', 'steps', 'residential']", 'ladder', "['pedestrian', 'steps', 'residential']", "['living_street', 'pedestrian']", "['tertiar

### 2. Exclude links with highway as an array

In [5]:
# Exclude arrays (porque solo son algunos cuantos)
mask_highway_is_array = links_with_atts['HIGHWAY'].astype(str).str.startswith("[")

links_highway_array = links_with_atts[mask_highway_is_array].copy()
print(f"Valid links that WON'T BE CONSIDERED for connector creation")
print(links_highway_array['HIGHWAY'].value_counts())
print()

#Keep only links with attributes & which highway value IS NOT an array
valid_links = links_with_atts[~mask_highway_is_array].copy()
print(f"Links with attributes and valid highway {valid_links['HIGHWAY'].value_counts()}")

Valid links that WON'T BE CONSIDERED for connector creation
HIGHWAY
['tertiary', 'trunk_link']         7
['tertiary_link', 'tertiary']      2
['motorway_link', 'secondary']     2
['secondary_link', 'secondary']    2
['primary_link', 'trunk_link']     1
['tertiary', 'secondary']          1
['motorway_link', 'tertiary']      1
['primary', 'trunk']               1
['motorway', 'trunk']              1
Name: count, dtype: int64

Links with attributes and valid highway HIGHWAY
residential       269597
service            73314
living_street      49634
tertiary           27874
unclassified       14134
secondary          10094
primary             7616
trunk               1414
primary_link         780
tertiary_link        735
trunk_link           555
secondary_link       510
motorway_link        493
motorway             373
Name: count, dtype: int64


### 3. Exclude links with highway type "X_link"
incorporaciones porque no queremos crear conectores a ellas

In [6]:
mask_incorporaciones = (
    valid_links['HIGHWAY']
    .astype(str)
    .str.endswith("_link", na=False)
)

#Links que son incorportaciones
links_incorporaciones = valid_links[mask_incorporaciones].copy()
print(f"From valid links {len(links_incorporaciones)} are incorporaciones")
print(links_incorporaciones['HIGHWAY'].value_counts())
print()

#Quedarnos con links normales que no son incorporaciones
main_links = valid_links[~mask_incorporaciones].copy()
print(f"Links base to create connectores are: {len(main_links)}")
print(main_links['HIGHWAY'].value_counts())

From valid links 3073 are incorporaciones
HIGHWAY
primary_link      780
tertiary_link     735
trunk_link        555
secondary_link    510
motorway_link     493
Name: count, dtype: int64

Links base to create connectores are: 454050
HIGHWAY
residential      269597
service           73314
living_street     49634
tertiary          27874
unclassified      14134
secondary         10094
primary            7616
trunk              1414
motorway            373
Name: count, dtype: int64


### 4. Assign priority to links based on their highway
vias principales tienen mayor prioridad que las locales

In [7]:
highway_priority_map = {
    "primary": 1,
    "secondary": 2,
    "tertiary": 3,
    "trunk": 4,
    "motorway": 5,
    "unclassified": 6,
    "residential": 7,
    "service": 8,
    "living_street": 9
}

#Asign priority to main links based on their highway type
main_links['PRIORITY'] = main_links['HIGHWAY'].map(highway_priority_map)

In [8]:
# Check that there are no links without priority
main_links.loc[
    main_links["PRIORITY"].isna(),
    "HIGHWAY"
].value_counts()

Series([], Name: count, dtype: int64)

### 5. Get from & to nodes
from main links with highway priority

In [9]:
from_nodes = main_links[
    ["FromNodeNo", "PRIORITY"]
].rename(columns={"FromNodeNo": "NodeNo"})

to_nodes = main_links[
    ["ToNodeNo", "PRIORITY"]
].rename(columns={"ToNodeNo": "NodeNo"})

In [10]:
nodes_from_links = pd.concat(
    [from_nodes, to_nodes],
    ignore_index=True
)

In [11]:
# For cases of the same node in two diff links. keep the highest (min) priority for that node
node_priorities = (
    nodes_from_links
    .groupby("NodeNo", as_index=False)
    .agg(PRIORITY=("PRIORITY", "min"))
)

In [12]:
node_priorities

,NodeNo,PRIORITY
0,1.0,5
1,2.0,5
2,3.0,5
3,4.0,4
4,5.0,4
...,...,...
201346,213443.0,8
201347,213451.0,8
201348,213452.0,8
201349,213453.0,8


### 6. Read nodes & Merge priority on nodes


In [13]:
#Read nodes from Visum
nodes = pd.DataFrame({
    "NodeNo": [i[1] for i in Visum.Net.Nodes.GetMultiAttValues("No")],
})

#Merge with node_priorities on=NodeNo
nodes = nodes.merge(
    node_priorities,
    on='NodeNo',
    how='left'
)

# Fill with 0 those nodes that were not found in node priorities 
# those are nodes that won't be considered for connector creation
nodes['PRIORITY'] = nodes['PRIORITY'].fillna(0).astype(int)

### 7. Create PRIORITY column in Visum 
as a tag column to know which nodes are candidates for connector creation

In [17]:
visum_nodes = Visum.Net.Nodes

nodes = nodes.reset_index(drop=True)
nodes.index = nodes.index + 1

node_priority = list(zip(
    nodes.index,
    nodes['PRIORITY'].astype(int)
))

visum_nodes.SetMultiAttValues("HIGHWAY_PRIORITY", node_priority)

In [18]:
nodes

,NodeNo,PRIORITY
1,1.0,5
2,2.0,5
3,3.0,5
4,4.0,4
5,5.0,4
...,...,...
213452,213452.0,8
213453,213453.0,8
213454,213454.0,8
213455,213455.0,0
